In [15]:
import pandas as pd
from rich import print as pprint

In [16]:
pa_record = pd.read_csv("pa_record.csv")
pa_record

,gameNo,awayTeam,homeTeam,team_type,inning,scored,batterName,batterHand,pitcherName,pitcherHand,...,homeScores,strikes,balls,outs,bases,result,RBI,locationCode,trajectory,hardness
0,1,樂天桃猿,味全龍,away,1,False,陳晨威,L,徐若熙,R,...,0,0,1,0,0,GO,0,4M,G,M
1,1,樂天桃猿,味全龍,away,1,False,林立,R,徐若熙,R,...,0,2,2,1,0,SO,0,NaN,NaN,NaN
2,1,樂天桃猿,味全龍,away,1,False,梁家榮,L,徐若熙,R,...,0,2,1,2,0,2B,0,9LF,G,H
3,1,樂天桃猿,味全龍,away,1,False,廖健富,L,徐若熙,R,...,0,0,1,2,2,1B,0,8,G,H
4,1,樂天桃猿,味全龍,away,2,False,朱育賢,L,徐若熙,R,...,0,2,0,0,0,FO,0,7LSF,F,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27595,360,富邦悍將,味全龍,home,10,False,拿莫．伊漾,R,富藍戈,R,...,2,2,1,2,2,FO,0,8,F,M
27596,360,富邦悍將,味全龍,home,11,False,拿莫．伊漾,R,富藍戈,R,...,2,0,0,0,0,NaN,0,NaN,NaN,NaN
27597,360,富邦悍將,味全龍,home,11,False,吉力吉撈．鞏冠,R,富藍戈,R,...,2,0,0,0,2,NaN,0,78S,G,M
27598,360,富邦悍將,味全龍,home,11,False,吉力吉撈．鞏冠,R,吳世豪,R,...,2,1,1,0,2,1B,0,78S,G,M


In [17]:
pa_record['batterName'].nunique()

173

In [18]:
# 定義計算打擊指標所需的變數
At_Bat = ['1B', '2B', '3B', 'HR', 'H', 'GO', 'FO', 'SO', 'E', 'FC', 'GIDP', 'GIDP_E', 'SF_E', 'SH_E', 'SH_FC', 'ID', 'IH']
Hit = ['1B', '2B', '3B', 'HR', 'H', 'IH']
On_Base = ['1B', '2B', '3B', 'HR', 'H', 'uBB', 'IBB', 'HBP']
At_Bat_and_On_Base = At_Bat + ['uBB', 'IBB', 'HBP', 'SF']
TB = {'1B': 1, '2B': 2, '3B': 3, 'HR': 4, 'IH': 4, 'H': 1}


In [19]:
import numpy as np

# 計算每個打者對左投和右投的OPS
def calculate_ops(df):
    """計算給定dataframe的OPS"""
    if len(df) == 0:
        return np.nan
    
    # 計算打數相關
    at_bat_count = len(df[df['result'].isin(At_Bat)])
    if at_bat_count == 0:
        return np.nan
    
    # 計算上壘率
    on_base_count = len(df[df['result'].isin(On_Base)])
    at_bat_and_on_base_count = len(df[df['result'].isin(At_Bat_and_On_Base)])
    
    if at_bat_and_on_base_count == 0:
        obp = 0
    else:
        obp = on_base_count / at_bat_and_on_base_count
    
    # 計算長打率
    total_bases = df['result'].apply(lambda x: TB.get(x, 0)).sum()
    slg = total_bases / at_bat_count
    
    # 計算OPS
    ops = obp + slg
    
    return round(ops, 4)

# 獲取所有打者名單
batters = pa_record['batterName'].unique()

# 儲存結果
results = []

for batter in batters:
    batter_data = pa_record[pa_record['batterName'] == batter]
    batter_hand = batter_data['batterHand'].iloc[0]
    # 對左投的數據
    vs_left = batter_data[batter_data['pitcherHand'] == 'L']
    ops_vs_left = calculate_ops(vs_left)
    pa_vs_left = len(vs_left)
    
    # 對右投的數據
    vs_right = batter_data[batter_data['pitcherHand'] == 'R']
    ops_vs_right = calculate_ops(vs_right)
    pa_vs_right = len(vs_right)
    
    results.append({
        'batterName': batter,
        'OPS_vs_LHP': ops_vs_left,
        'PA_vs_LHP': pa_vs_left,
        'OPS_vs_RHP': ops_vs_right,
        'PA_vs_RHP': pa_vs_right,
        'Total_PA': pa_vs_left + pa_vs_right,
        'batterHand': batter_hand,
    })

# 建立結果DataFrame
batter_ops_df = pd.DataFrame(results)
batter_ops_df = batter_ops_df.sort_values('Total_PA', ascending=False)

print(f"總共有 {len(batter_ops_df)} 位打者")
batter_ops_df


總共有 173 位打者


,batterName,OPS_vs_LHP,PA_vs_LHP,OPS_vs_RHP,PA_vs_RHP,Total_PA,batterHand
60,邱智呈,0.7950,209,0.8783,326,535,L
0,陳晨威,0.7152,176,0.8157,356,532,L
31,曾子祐,0.6532,130,0.6877,394,524,R
12,吉力吉撈．鞏冠,0.9071,166,0.7783,347,513,R
20,岳政華,0.5827,138,0.6686,374,512,L
...,...,...,...,...,...,...,...
155,蕭憶銘,0.0000,1,NaN,0,1,R
147,林岳谷,NaN,0,0.0000,1,1,R
145,李聖裕,NaN,0,0.0000,1,1,L
134,周委宏,NaN,0,0.0000,1,1,L


In [61]:
# 查看OPS差異（對右投OPS - 對左投OPS）
batter_ops_df['OPS_Diff_R_minus_L'] = batter_ops_df['OPS_vs_RHP'] - batter_ops_df['OPS_vs_LHP']
batter_ops_df['PA'] = batter_ops_df['PA_vs_LHP'] + batter_ops_df['PA_vs_RHP']
# 篩選有效樣本（左右投都至少30打席）
pa=60
# qualified_batters = batter_ops_df[(batter_ops_df['PA_vs_LHP'] >= pa) & (batter_ops_df['PA_vs_RHP'] >= pa)].copy()
qualified_batters = batter_ops_df[(batter_ops_df['PA']>=240)&(batter_ops_df['PA_vs_LHP']>=pa)&(batter_ops_df['PA_vs_RHP']>=pa)].copy()
# 建立對右投表現較好的打者dataframe（前10名）
better_vs_RHP_df = qualified_batters.nlargest(10, 'OPS_Diff_R_minus_L')[['batterName', 'batterHand', 'OPS_vs_RHP', 'OPS_vs_LHP', 'OPS_Diff_R_minus_L', 'PA_vs_RHP', 'PA_vs_LHP', 'PA']].reset_index(drop=True)

# 建立對左投表現較好的打者dataframe（前10名）
better_vs_LHP_df = qualified_batters.nsmallest(10, 'OPS_Diff_R_minus_L')[['batterName', 'batterHand', 'OPS_vs_RHP', 'OPS_vs_LHP', 'OPS_Diff_R_minus_L', 'PA_vs_RHP', 'PA_vs_LHP', 'PA']].reset_index(drop=True)

print(f"對右投表現較好的前10名打者（左右投都至少{pa}打席）：")
better_vs_RHP_df




對右投表現較好的前10名打者（左右投都至少60打席）：


,batterName,batterHand,OPS_vs_RHP,OPS_vs_LHP,OPS_Diff_R_minus_L,PA_vs_RHP,PA_vs_LHP,PA
0,郭嚴文,L,0.8011,0.5122,0.2889,207,70,277
1,高宇杰,R,0.7000,0.4825,0.2175,245,85,330
2,朱育賢,L,0.8252,0.6400,0.1852,310,135,445
3,李凱威,L,0.8153,0.6637,0.1516,338,161,499
4,林泓育,R,0.8563,0.7185,0.1378,254,124,378
5,王正棠,L,0.8793,0.7449,0.1344,247,109,356
6,廖健富,L,0.8262,0.6931,0.1331,200,91,291
7,王威晨,L,0.8011,0.6844,0.1167,232,70,302
8,陳子豪,L,0.9030,0.7875,0.1155,245,100,345
9,陳文杰,L,0.7334,0.6268,0.1066,346,120,466


In [62]:
print(f"\n對左投表現較好的前10名打者（左右投都至少{pa}打席）：")
better_vs_LHP_df


對左投表現較好的前10名打者（左右投都至少60打席）：


,batterName,batterHand,OPS_vs_RHP,OPS_vs_LHP,OPS_Diff_R_minus_L,PA_vs_RHP,PA_vs_LHP,PA
0,張政禹,L,0.5773,0.8142,-0.2369,161,88,249
1,江坤宇,R,0.6415,0.8689,-0.2274,288,86,374
2,王博玄,L,0.6382,0.8217,-0.1835,304,106,410
3,陳重廷,R,0.5377,0.6764,-0.1387,239,164,403
4,吉力吉撈．鞏冠,R,0.7783,0.9071,-0.1288,347,166,513
5,范國宸,R,0.5467,0.6658,-0.1191,225,98,323
6,張肇元,R,0.6482,0.7632,-0.1150,240,96,336
7,陳俊秀,R,0.7722,0.8843,-0.1121,280,102,382
8,許哲晏,R,0.5708,0.6782,-0.1074,166,91,257
9,陳傑憲,L,0.7948,0.9015,-0.1067,292,181,473


In [ ]:
# # 將完整的打者OPS數據儲存為CSV
# batter_ops_df.to_csv('batter_ops_vs_pitcher_hand.csv', index=False, encoding='utf-8-sig')
# print("✓ 已儲存完整數據至 batter_ops_vs_pitcher_hand.csv")

# # 儲存對右投表現較好的打者
# better_vs_RHP_df.to_csv('better_vs_RHP_batters.csv', index=False, encoding='utf-8-sig')
# print("✓ 已儲存對右投表現較好的打者至 better_vs_RHP_batters.csv")

# # 儲存對左投表現較好的打者
# better_vs_LHP_df.to_csv('better_vs_LHP_batters.csv', index=False, encoding='utf-8-sig')
# print("✓ 已儲存對左投表現較好的打者至 better_vs_LHP_batters.csv")


✓ 已儲存完整數據至 batter_ops_vs_pitcher_hand.csv
✓ 已儲存對右投表現較好的打者至 better_vs_RHP_batters.csv
✓ 已儲存對左投表現較好的打者至 better_vs_LHP_batters.csv


In [60]:
# 顯示所有173位打者的完整數據
print(f"=== 所有{len(batter_ops_df)}位打者對左右投的OPS ===")

batter_ops_df[(batter_ops_df['PA']>=240)].sort_values('OPS_Diff_R_minus_L', ascending=False)

=== 所有173位打者對左右投的OPS ===


,batterName,OPS_vs_LHP,PA_vs_LHP,OPS_vs_RHP,PA_vs_RHP,Total_PA,batterHand,OPS_Diff_R_minus_L,PA
15,郭嚴文,0.5122,70,0.8011,207,277,L,0.2889,277
27,高宇杰,0.4825,85,0.7000,245,330,R,0.2175,330
4,朱育賢,0.6400,135,0.8252,310,445,L,0.1852,445
13,李凱威,0.6637,161,0.8153,338,499,L,0.1516,499
10,林泓育,0.7185,124,0.8563,254,378,R,0.1378,378
48,王正棠,0.7449,109,0.8793,247,356,L,0.1344,356
3,廖健富,0.6931,91,0.8262,200,291,L,0.1331,291
21,王威晨,0.6844,70,0.8011,232,302,L,0.1167,302
23,陳子豪,0.7875,100,0.9030,245,345,L,0.1155,345
32,陳文杰,0.6268,120,0.7334,346,466,L,0.1066,466


In [25]:
# 基本統計資訊
print("=== 打者對左右投OPS統計摘要 ===\n")
print(f"總打者數：{len(batter_ops_df)}")
print(f"有對左投數據的打者數：{batter_ops_df['OPS_vs_LHP'].notna().sum()}")
print(f"有對右投數據的打者數：{batter_ops_df['OPS_vs_RHP'].notna().sum()}")
print(f"左右投都有數據的打者數：{((batter_ops_df['OPS_vs_LHP'].notna()) & (batter_ops_df['OPS_vs_RHP'].notna())).sum()}")

print("\n=== OPS平均值（去除NaN） ===")
print(f"平均對左投OPS：{batter_ops_df['OPS_vs_LHP'].mean():.4f}")
print(f"平均對右投OPS：{batter_ops_df['OPS_vs_RHP'].mean():.4f}")

print("\n=== 打席數分布 ===")
print(f"總打席數：{batter_ops_df['Total_PA'].sum()}")
print(f"對左投總打席數：{batter_ops_df['PA_vs_LHP'].sum()}")
print(f"對右投總打席數：{batter_ops_df['PA_vs_RHP'].sum()}")


=== 打者對左右投OPS統計摘要 ===

總打者數：173
有對左投數據的打者數：156
有對右投數據的打者數：172
左右投都有數據的打者數：155

=== OPS平均值（去除NaN） ===
平均對左投OPS：0.5985
平均對右投OPS：0.5778

=== 打席數分布 ===
總打席數：27600
對左投總打席數：8347
對右投總打席數：19253
